# Avaliação dos Modelos de Fatoração de Matrizes

Objetivo: Implementar e avaliar os modelo ALS

### Importando as bibliotecas

In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import sys
import os
sys.path.append(os.path.abspath("../src"))
from data_processing import load_and_prepare_data, split_train_validation, create_mappings
from evaluation import evaluate_recommender


### Carregando e preparando os dados

In [2]:
df = load_and_prepare_data("../data/processed/online_retail_clean.parquet")
train_df, validation_df = split_train_validation(df)

# Criando os mapeamentos de IDs para índices
user_map, item_map, inverse_user_map, inverse_item_map = create_mappings(train_df)

# Agrupar para contar quantas vezes cada usuário comprou cada item
user_item_counts = (
    train_df.groupby(['CustomerID', 'StockCode'])
    .size()
    .reset_index(name='count')
)

# Mapear índices dos usuários e itens a partir do dataframe agrupado
user_indices = user_item_counts['CustomerID'].map(user_map)
item_indices = user_item_counts['StockCode'].map(item_map)
counts = user_item_counts['count']

num_users = len(user_map)
num_items = len(item_map)

# Criando a matriz esparsa de contagens, servirá como matriz de confiança
user_item_matrix_counts = csr_matrix(
    (counts, (user_indices, item_indices)),
    shape=(num_users, num_items)
)

# --- Verificação Final ---
print("\nMatriz de interações (contagens) criada com sucesso!")
print(f"Dimensões da matriz: {user_item_matrix_counts.shape}")
print(f"Total de interações na matriz (não-zeros): {user_item_matrix_counts.nnz}")


Matriz de interações (contagens) criada com sucesso!
Dimensões da matriz: (2843, 3608)
Total de interações na matriz (não-zeros): 196797


### Treinando o modelo

In [3]:
import implicit

model_als = implicit.als.AlternatingLeastSquares(
    factors=50,
    regularization=0.01,
    iterations=100,
    use_gpu=False,
    random_state=42
)

model_als.fit(user_item_matrix_counts)



c:\Users\DiegoGonçalvesMota\Documents\tcc\recommender-system\venv\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/100 [00:00<?, ?it/s]

In [4]:
# ===============================================
# --- 3. Gerar Recomendações (Exemplo) ---
# Pegar o primeiro usuario como exemplo
user_id_example = list(user_map.keys())[5]
user_idx_example = user_map[user_id_example]

# A função recommend pede o ÍNDICE do usuário e a matriz esparsa ORIGINAL (usuários x itens)
# Ela já filtra os itens que o usuário comprou
recommended_indices, scores = model_als.recommend(
    user_idx_example,
    user_item_matrix_counts[user_idx_example]
)

# Traduzir de volta para StockCodes
recommended_items = [inverse_item_map[i] for i in recommended_indices]
print(f"Itens que o usuário {user_id_example} já comprou:")

user_bought_items = train_df[train_df['CustomerID'] == user_id_example][['StockCode', 'Description']].drop_duplicates()
print(user_bought_items)

print(f"Recomendações para o usuário {user_id_example}:")
recommended_items_df = pd.DataFrame({
    'StockCode': recommended_items,
    'Description': [train_df[train_df['StockCode'] == code]['Description'].iloc[0] for code in recommended_items],
    'Score': scores
})
print(recommended_items_df)

Itens que o usuário 15291 já comprou:
       StockCode                          Description
82         22114    HOT WATER BOTTLE TEA AND SYMPATHY
83         21733     RED HANGING HEART T-LIGHT HOLDER
22794      82484    WOOD BLACK BOARD ANT WHITE FINISH
22795      82486    WOOD S/3 CABINET ANT WHITE FINISH
22796      21754             HOME BUILDING BLOCK WORD
...          ...                                  ...
161433     21670        BLUE SPOT CERAMIC DRAWER KNOB
319337     22578   WOODEN STAR CHRISTMAS SCANDINAVIAN
319338     22579   WOODEN TREE CHRISTMAS SCANDINAVIAN
319339     22577  WOODEN HEART CHRISTMAS SCANDINAVIAN
324661     22112           CHOCOLATE HOT WATER BOTTLE

[61 rows x 2 columns]
Recomendações para o usuário 15291:
  StockCode                         Description     Score
0     48187                 DOORMAT NEW ENGLAND  0.854662
1    85123A  WHITE HANGING HEART T-LIGHT HOLDER  0.832907
2     23284       DOORMAT KEEP CALM AND COME IN  0.552276
3     21671        RED 

###  Criando a matriz de validação

In [5]:
# 1. Preparar a matriz de validação (ground truth)
validation_user_indices = validation_df['CustomerID'].map(user_map)
validation_item_indices = validation_df['StockCode'].map(item_map)

# Removendo interações de usuários ou itens que não estavam no treino
valid_interactions = ~ (validation_user_indices.isna() | validation_item_indices.isna())
validation_user_indices = validation_user_indices[valid_interactions].astype(int)
validation_item_indices = validation_item_indices[valid_interactions].astype(int)

validation_matrix = csr_matrix(
    (np.ones(len(validation_user_indices)), (validation_user_indices, validation_item_indices)),
    shape=user_item_matrix_counts.shape
)

### Função de recomendação do ALS

In [6]:
def recommend_als_items(user_id, K, exclude):
    user_idx = user_map.get(user_id)

    if user_idx is None:
        return []
    
    recommended_indices, _ = model_als.recommend(
        user_idx,
        user_item_matrix_counts[user_idx],
        N=K,
    )

    recommended_items = [inverse_item_map[i] for i in recommended_indices]
    return recommended_items
    

### Avaliando o ALS

In [7]:
print("\n--- 5. Avaliando o modelo ALS usando o framework de 'src' ---")

K_values = [5, 10, 20]
all_results_als = []

train_user_items = (train_df.drop_duplicates(['CustomerID','InvoiceNo','StockCode'])
                    .groupby('CustomerID')['StockCode'].apply(set).to_dict())

validation_user_items = (validation_df.drop_duplicates(['CustomerID','InvoiceNo','StockCode'])
                         .groupby('CustomerID')['StockCode'].apply(set).to_dict())

validation_users = list(validation_user_items.keys())
valid_users_for_eval = [u for u in validation_users if u in user_map]

# Loop para avaliar em diferentes valores de K
for k in K_values:
    print(f"\nCalculando métricas para K={k}...")

    results_k = evaluate_recommender(
        users=valid_users_for_eval,
        recommend_fn=recommend_als_items,   # A função de recomendação do ALS
        truth_dict=validation_user_items,   # O gabarito (validação)
        exclude_seen_dict=train_user_items, # Itens para excluir (treino)
        k=k                                 # O valor de K atual
    )
    
    row = {
        'K': k,
        'users_eval': results_k['users_eval'],
        'Precision': results_k[f'P@{k}'],
        'Recall': results_k[f'R@{k}'],
        'NDCG': results_k[f'NDCG@{k}']
    }
    all_results_als.append(row)
    print(f"Resultados para K={k}: {results_k}")

# Apresentação Final dos Resultados
results_df_als = pd.DataFrame(all_results_als).set_index('K')
print("\n\n--- Resultados Finais do Modelo ALS ---")
print(results_df_als)


--- 5. Avaliando o modelo ALS usando o framework de 'src' ---

Calculando métricas para K=5...
Resultados para K=5: {'users_eval': 2843, 'P@5': 0.0632430531129089, 'R@5': 0.019316677569214083, 'NDCG@5': 0.06796234073801365}

Calculando métricas para K=10...
Resultados para K=10: {'users_eval': 2843, 'P@10': 0.05550474850510025, 'R@10': 0.03392883683396217, 'NDCG@10': 0.06472247074716311}

Calculando métricas para K=20...
Resultados para K=20: {'users_eval': 2843, 'P@20': 0.04560323601829054, 'R@20': 0.05388136363188242, 'NDCG@20': 0.06368967676914937}


--- Resultados Finais do Modelo ALS ---
    users_eval  Precision    Recall      NDCG
K                                            
5         2843   0.063243  0.019317  0.067962
10        2843   0.055505  0.033929  0.064722
20        2843   0.045603  0.053881  0.063690


### Comparando o ALS com o KNN

In [8]:
def get_results_df(results, model_name):
    """
    Converte a lista de resultados em um DataFrame e adiciona o nome do modelo.
    """
    df_results = pd.DataFrame(results)
    df_results['Model'] = model_name
    return df_results

def compare_models(df_model_A, df_model_B, model_A_name, model_B_name, metric_cols=['Precision', 'Recall', 'NDCG']):
    """
    Compara as métricas de dois modelos e imprime o resultado formatado
    com a melhoria percentual.
    """
    print(f"--- Comparação: {model_A_name} vs. {model_B_name} ---\n")
    
    # Junta os dois dataframes pela coluna 'K'
    comparison_df = pd.merge(df_model_A, df_model_B, on='K', suffixes=(f'_{model_A_name}', f'_{model_B_name}'))
    
    for k_value in comparison_df['K'].unique():
        print(f"K={k_value}\n")
        
        row = comparison_df[comparison_df['K'] == k_value].iloc[0]
        
        for metric in metric_cols:
            val_A = row[f'{metric}_{model_A_name}']
            val_B = row[f'{metric}_{model_B_name}']
            
            # Calcula a melhoria percentual (uplift)
            uplift = ((val_A - val_B) / val_B) * 100 if val_B > 0 else float('inf')
            
            # Formata a string de saída
            print(f"- {metric}@{k_value}: {val_A:.4f} vs {val_B:.4f} → {uplift:+.0f}%")
        
        print("-" * 20)

df_knn = pd.read_parquet("../data/processed/df_knn.parquet")
df_als = get_results_df(all_results_als, "ALS")

compare_models(df_als, df_knn, "ALS", "KNN")


--- Comparação: ALS vs. KNN ---

K=5

- Precision@5: 0.0632 vs 0.0428 → +48%
- Recall@5: 0.0193 vs 0.0116 → +66%
- NDCG@5: 0.0680 vs 0.0444 → +53%
--------------------
K=10

- Precision@10: 0.0555 vs 0.0378 → +47%
- Recall@10: 0.0339 vs 0.0198 → +72%
- NDCG@10: 0.0647 vs 0.0420 → +54%
--------------------
K=20

- Precision@20: 0.0456 vs 0.0318 → +43%
- Recall@20: 0.0539 vs 0.0327 → +65%
- NDCG@20: 0.0637 vs 0.0413 → +54%
--------------------
